In [1]:
# import required python modules
import os
import re
import unicodedata
import fitz
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
in_path = os.path.join(base_dir, "task3_capmas_private_industrial_production_2021.pdf")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# open raw pdf document
doc = fitz.open(in_path)
page_rows = []
index_rows = []

In [1]:
# inspect and clean pages
for idx, page in enumerate(doc):
    raw_text = unicodedata.normalize("NFKC", page.get_text("text"))
    clean_text = re.sub(r"[\x00-\x1f]", " ", raw_text)
    lines = [ln.strip() for ln in clean_text.splitlines() if ln.strip()]
    imgs = len(page.get_images())
    ptype = "picture only" if not lines and imgs > 0 else "table or text"
    page_rows.append({"page": idx + 1, "line_count": len(lines), "image_count": imgs, "page_type": ptype, "sample_text": " ".join(lines[:3])})
    if 5 <= idx + 1 <= 10:
        for ln in lines:
            if "Table" in ln or "Section" in ln:
                index_rows.append({"page": idx + 1, "index_entry": ln})

In [1]:
# convert lists to dataframes
df_pages = pd.DataFrame(page_rows)
df_index = pd.DataFrame(index_rows)

In [1]:
# save cleaned pdf outputs
csv_out = os.path.join(out_dir, "cleaned_task3_capmas_private_industrial_production_2021_pdf.csv")
df_pages.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
xlsx_out = os.path.join(out_dir, "cleaned_task3_capmas_private_industrial_production_2021_pdf.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    df_index.to_excel(writer, sheet_name="bulletin_index", index=False)
    df_pages.to_excel(writer, sheet_name="page_summary", index=False)